# Guide intelligent du Musée · 04 ResNet (jalon `m3`)

Partir d'un ResNet-18 pré-entraîné sur ImageNet, n'entraîner que sa tête puis affiner son dernier bloc, et le comparer au réseau dense sur les critères des bornes : précision, vitesse sur CPU, taille.

Contexte du projet, données et découpage : voir `01_donnees.ipynb`.

## 1. Mise en place

In [ ]:
import copy
import math
import platform
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader

from commun import (IMAGENET_MEAN, IMAGENET_STD, MODELS_DIR, SEED, ArtworkDataset, ResNetClassifier, fit,
                    get_device, load_dense, load_images, load_metadata, load_resnet, normalization_stats,
                    predict, save_resnet, set_seed, split_indices, standardize)

set_seed()
DEVICE = get_device()
meta, CLASSES = load_metadata()
X_all, y_all = load_images(meta)
splits = split_indices(meta)
MEAN, STD = normalization_stats(X_all, splits["train"])

print(f"PyTorch {torch.__version__}, calculs sur : {DEVICE}")
print(f"{len(meta)} œuvres, {len(CLASSES)} styles ; " + ", ".join(f"{k} : {len(i)}" for k, i in splits.items()))

## 2. Tête seule

Recette de la séance 7 : un ResNet-18 pré-entraîné sur ImageNet, dont la dernière couche (1 000 classes) est remplacée par une nouvelle tête à 5 sorties, seule entraînée. Les entrées sont préparées comme au pré-entraînement : agrandies de 128 à 224 pixels, puis normalisées avec la moyenne et l'écart-type d'ImageNet, pas avec ceux du train.

Les parties gelées restent en mode évaluation pendant l'entraînement : leurs batch norms gardent les statistiques d'ImageNet au lieu de se recalculer sur nos batchs, sinon le backbone ne serait pas vraiment gelé.

### 2.1 Modèle et paramètres

In [ ]:
N_EPOCHS = 20
BATCH_SIZE = 64
LR_HEAD = 1e-3
LN_C = math.log(len(CLASSES))  # perte d'un modèle qui répond au hasard : ln 5

datasets = {k: ArtworkDataset(X_all, y_all, splits[k], IMAGENET_MEAN, IMAGENET_STD) for k in ("train", "val")}
val_loader = DataLoader(datasets["val"], batch_size=256)
loss_fn = nn.CrossEntropyLoss()

set_seed()
resnet = ResNetClassifier(len(CLASSES)).to(DEVICE)
n_params = sum(p.numel() for p in resnet.parameters())
n_trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)

print(f"Nouvelle tête : {resnet.net.fc}")
print(f"Paramètres au total : {n_params:,}".replace(",", " "))
print(f"Paramètres entraînables : {n_trainable:,}".replace(",", " ") + f", soit {100 * n_trainable / n_params:.2f} %")

### 2.2 Entraînement de la tête

Mêmes splits et même boucle (`fit`) que le réseau dense. Adam avec un taux d'apprentissage de 1e-3 sur la tête seule, batch de 64, 20 époques, graine 0. Le test n'est pas utilisé.

In [ ]:
def summarize(history: dict[str, list[float]]) -> dict[str, float]:
    """Valeurs à la dernière époque et minimum de la perte de validation."""
    best = int(np.argmin(history["val_loss"]))
    return {"perte_train": history["train_loss"][-1], "perte_val": history["val_loss"][-1],
            "acc_train": history["train_acc"][-1], "acc_val": history["val_acc"][-1],
            "epoque_min_perte_val": best + 1, "min_perte_val": history["val_loss"][best]}


train_loader = DataLoader(datasets["train"], batch_size=BATCH_SIZE, shuffle=True)
optimizer = torch.optim.Adam(resnet.net.fc.parameters(), lr=LR_HEAD)
t0 = time.perf_counter()
head_history = fit(resnet, train_loader, val_loader, loss_fn, optimizer, N_EPOCHS, DEVICE)
epoch_s = (time.perf_counter() - t0) / N_EPOCHS
head = summarize(head_history)

print(f"Dernière époque : accuracy {head['acc_train']:.3f} en train, {head['acc_val']:.3f} en validation "
      f"(écart {head['acc_train'] - head['acc_val']:.3f}) ; perte {head['perte_train']:.3f} en train, "
      f"{head['perte_val']:.3f} en validation")
print(f"Minimum de la perte de validation à l'époque {head['epoque_min_perte_val']} ({head['min_perte_val']:.3f})")
print(f"Meilleure accuracy de validation : {max(head_history['val_acc']):.3f}, "
      f"à l'époque {int(np.argmax(head_history['val_acc'])) + 1}")
print(f"Durée d'une époque : {epoch_s:.1f} s, calculs sur {DEVICE} ({platform.platform()})")

### 2.3 Courbes de suivi

In [ ]:
# Palette de référence (créneaux 1 et 2), encre et grille discrètes, comme dans 03_diagnostic
COLORS = {"train": "#2a78d6", "val": "#eb6834"}
LABELS = {"train": "train", "val": "validation"}
INK, MUTED = "#52514e", "#898781"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.titlesize": 11, "font.size": 10,
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 1.5,
})


def plot_curves(ax, history: dict[str, list[float]], metric: str, chance: float) -> None:
    """Courbes train et validation d'une métrique ('loss' ou 'acc'), étiquetées en bout de ligne,
    avec en pointillés la valeur d'un modèle qui répond au hasard."""
    epochs = np.arange(1, len(history[f"train_{metric}"]) + 1)
    ends = {split: history[f"{split}_{metric}"][-1] for split in ("train", "val")}
    low, high = sorted(ends, key=ends.get)
    for split in ("train", "val"):
        ax.plot(epochs, history[f"{split}_{metric}"], color=COLORS[split], label=LABELS[split])
        nudge = -6 if split == low else 6  # écarte les deux étiquettes quand les courbes finissent proches
        ax.annotate(LABELS[split], (epochs[-1], ends[split]), xytext=(5, nudge), textcoords="offset points",
                    va="center", color=INK, fontsize=9)
    ax.axhline(chance, color=MUTED, linestyle="--", linewidth=1, label="réponse au hasard")
    ax.set_xlabel("époque")
    ax.set_xlim(1, epochs[-1] * 1.18)  # place pour les étiquettes en bout de ligne
    ax.set_xticks([1, *range(5, epochs[-1] + 1, 5)])


fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(11, 3.8))
plot_curves(ax_loss, head_history, "loss", LN_C)
ax_loss.set_ylabel("perte (entropie croisée)")
ax_loss.set_title("Perte")
plot_curves(ax_acc, head_history, "acc", 1 / len(CLASSES))
ax_acc.set_ylabel("accuracy")
ax_acc.set_title("Accuracy")
fig.legend(*ax_loss.get_legend_handles_labels(), loc="upper right", ncol=3, frameon=False)
fig.suptitle("ResNet-18, tête seule, par époque", x=0.01, ha="left", fontsize=12)
fig.tight_layout()
plt.show()

- **La tête seule apprend vite, avec très peu de paramètres** : en n'entraînant que 2 565 paramètres, l'accuracy de validation atteint 0,780 à la dernière époque, et 0,803 au mieux (époque 16). Les caractéristiques apprises sur ImageNet (contours, textures, motifs) séparent déjà bien les styles : une couche linéaire posée dessus suffit.
- **Peu de sur-apprentissage** : à la dernière époque, 0,863 en train contre 0,780 en validation, un écart de 0,083. La perte de validation est minimale à l'époque 17 (0,608) et reste presque plate ensuite : 20 époques suffisent.
- **C'est très au-dessus du réseau dense et de la régression logistique** de `03_diagnostic`. La section 4 les compare précisément, sur les modèles rechargés.
- **Coût** : environ 9,5 s par époque sur le GPU d'un Mac M1 (MPS), car chaque image traverse tout le backbone à chaque époque, même gelé.

### 2.4 Bilan

- **ResNet-18, tête seule : 0,780 d'accuracy de validation** à la dernière époque (graine 0), en n'entraînant que 2 565 des 11 179 077 paramètres (0,02 %).
- **Courbes saines** : pas de divergence, peu de sur-apprentissage (écart de 0,083), perte de validation minimale à l'époque 17.
- **Suite** (section 3) : dégeler `layer4` avec un taux d'apprentissage plus faible, pour adapter aux tableaux les motifs de haut niveau appris sur des photos.

## 3. Fine-tuning de `layer4`

On repart de la tête entraînée à la section 2 et on dégèle aussi le dernier bloc, `layer4`, avec un taux d'apprentissage dix fois plus faible que celui de la tête (1e-4 contre 1e-3) : les poids pré-entraînés n'ont besoin que de petits ajustements. Les batch norms de `layer4` repassent en mode entraînement.

Ce bloc compte des millions de paramètres pour 900 œuvres de train : l'entraînement s'arrête quand la perte de validation n'a pas baissé depuis 3 époques (15 au plus), et le modèle reprend les poids de sa meilleure époque.

Pour ne changer qu'une chose, le témoin est la tête seule entraînée plus longtemps, depuis le même point de départ et avec le même arrêt précoce. Trois graines pour chacun, car l'écart attendu est faible.

### 3.1 Fine-tuning et témoin

In [ ]:
N_EPOCHS_FT = 15
PATIENCE = 3
LR_LAYER4 = 1e-4
SEEDS = [0, 1, 2]
head_state = copy.deepcopy(resnet.state_dict())


def continue_training(seed: int, blocks: tuple[str, ...]) -> dict[str, list[float]]:
    """Repart de la tête entraînée et entraîne les blocs donnés, avec arrêt précoce sur la perte de validation."""
    resnet.load_state_dict(head_state)
    resnet.set_trainable(blocks)
    set_seed(seed)
    train_loader = DataLoader(datasets["train"], batch_size=BATCH_SIZE, shuffle=True)
    groups = [{"params": resnet.net.fc.parameters(), "lr": LR_HEAD}]
    if "layer4" in blocks:
        groups.append({"params": resnet.net.layer4.parameters(), "lr": LR_LAYER4})
    optimizer = torch.optim.Adam(groups)
    return fit(resnet, train_loader, val_loader, loss_fn, optimizer, N_EPOCHS_FT, DEVICE, patience=PATIENCE)


resnet.set_trainable(("layer4", "fc"))
print(f"Paramètres entraînables avec layer4 : {sum(p.numel() for p in resnet.parameters() if p.requires_grad):,}"
      .replace(",", " "))

RUNS = {"tête seule prolongée (témoin)": ("fc",), "fine-tuning de layer4": ("layer4", "fc")}
rows, ft_histories = [], {}
for name, blocks in RUNS.items():
    for seed in SEEDS:
        t0 = time.perf_counter()
        h = continue_training(seed, blocks)
        best = h["best_epoch"] - 1
        rows.append({"modele": name, "graine": seed, "acc_train": h["train_acc"][best], "acc_val": h["val_acc"][best],
                     "meilleure_epoque": h["best_epoch"], "s_par_epoque": (time.perf_counter() - t0) / len(h["val_acc"])})
        if "layer4" in blocks:
            ft_histories[seed] = h
            if seed == SEED:
                ft_state = copy.deepcopy(resnet.state_dict())

runs = pd.DataFrame(rows)
summary = runs.groupby("modele", sort=False).agg(
    acc_train=("acc_train", "mean"), acc_val=("acc_val", "mean"), acc_val_std=("acc_val", "std"),
    meilleure_epoque=("meilleure_epoque", "mean"), s_par_epoque=("s_par_epoque", "mean"))
ft_seed0 = runs.query("modele == 'fine-tuning de layer4' and graine == @SEED").iloc[0]

print(f"Tête seule (section 2, graine {SEED}, 20 époques) : {head['acc_val']:.3f} en validation")
for name, row in summary.iterrows():
    print(f"{name} : {row['acc_val']:.3f} ± {row['acc_val_std']:.3f} en validation sur {len(SEEDS)} graines")
print(f"Fine-tuning, graine {SEED} (modèle retenu) : {ft_seed0['acc_val']:.3f} en validation, "
      f"{ft_seed0['acc_train']:.3f} en train, meilleure époque {ft_seed0['meilleure_epoque']}")
print(f"Calculs sur {DEVICE} ({platform.platform()})")
summary.round(3)

### 3.2 Courbes de suivi

La tête seule de la section 2 (époques 1 à 20), puis le fine-tuning de `layer4` avec la graine 0, jusqu'à l'arrêt précoce. Le modèle retenu reprend les poids de la meilleure époque, marquée d'un point.

In [ ]:
ft_history = ft_histories[SEED]
full_history = {key: head_history[key] + ft_history[key] for key in head_history}
best_epoch = N_EPOCHS + ft_history["best_epoch"]

fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(11, 3.8))
plot_curves(ax_loss, full_history, "loss", LN_C)
ax_loss.set_ylabel("perte (entropie croisée)")
ax_loss.set_title("Perte")
plot_curves(ax_acc, full_history, "acc", 1 / len(CLASSES))
ax_acc.set_ylabel("accuracy")
ax_acc.set_title("Accuracy")
for ax, y in ((ax_loss, 0.8), (ax_acc, 0.35)):  # hauteurs choisies là où aucune courbe ne passe
    ax.axvline(N_EPOCHS + 0.5, color=MUTED, linewidth=1)
    ax.annotate("dégel de layer4", (N_EPOCHS + 0.5, y), xycoords=("data", "axes fraction"), xytext=(-4, 0),
                textcoords="offset points", ha="right", va="center", color=INK, fontsize=9)
ax_loss.plot(best_epoch, full_history["val_loss"][best_epoch - 1], "o", color=COLORS["val"], markersize=6)
ax_acc.plot(best_epoch, full_history["val_acc"][best_epoch - 1], "o", color=COLORS["val"], markersize=6)
fig.legend(*ax_loss.get_legend_handles_labels(), loc="upper right", ncol=3, frameon=False)
fig.suptitle("ResNet-18 : tête seule puis fine-tuning de layer4, par époque (graine 0)", x=0.01, ha="left", fontsize=12)
fig.tight_layout()
plt.show()

- **Le fine-tuning de `layer4` bat la tête seule** : 0,838 ± 0,015 en validation sur 3 graines, contre 0,780 pour la tête seule de la section 2 et 0,799 ± 0,008 pour le témoin, la même tête entraînée plus longtemps. L'écart avec le témoin dépasse nettement la dispersion entre graines : le gain vient bien du dégel de `layer4`, pas des époques supplémentaires.
- **Mais il sur-apprend très vite** : 8 396 293 paramètres entraînables pour 900 œuvres de train. Avec la graine 0, la meilleure époque est la 4ᵉ du fine-tuning, avec 0,997 en train contre 0,837 en validation. Sur les courbes, la perte de train tombe presque à zéro en trois époques, tandis que celle de validation ne baisse que brièvement avant de remonter. L'arrêt précoce, qui garde les poids de la meilleure époque, est indispensable.
- **Coût de l'entraînement** : une époque de fine-tuning dure plus longtemps qu'une époque de la tête seule (colonne `s_par_epoque`), car le gradient traverse maintenant `layer4`.

### 3.3 Sauvegarde

Le modèle fine-tuné de la graine 0 est sauvegardé avec sa configuration dans `modeles/`, qui n'est pas versionné, pour la comparaison et pour l'évaluation finale de `06_recommandation`.

In [ ]:
CPU = torch.device("cpu")
resnet.load_state_dict(ft_state)
resnet = resnet.to(CPU).eval()
MODELS_DIR.mkdir(exist_ok=True)
resnet_path = MODELS_DIR / "resnet.pt"
save_resnet(resnet, resnet_path)

reloaded = load_resnet(resnet_path)
assert torch.equal(predict(reloaded, val_loader, CPU), predict(resnet, val_loader, CPU))
print(f"{resnet_path.name} : {resnet_path.stat().st_size / 1e6:.1f} Mo, configuration {reloaded.config}")
print("Prédictions identiques après rechargement")

## 4. Comparaison avec le réseau dense

Les trois modèles sont rechargés depuis `modeles/`, sans être réentraînés, et comparés sur CPU, comme sur une borne :

- accuracy de validation, avec son intervalle de confiance à 95 % sur 300 œuvres ;
- temps d'inférence pour une œuvre, prétraitement compris (normalisation et, pour le ResNet, agrandissement à 224 pixels), médiane sur 100 appels, avec tous les cœurs du processeur puis avec un seul, plus proche d'une borne modeste ;
- taille du modèle, en paramètres et en Mo sur disque.

In [ ]:
logreg_path, dense_path = MODELS_DIR / "regression_logistique.joblib", MODELS_DIR / "reseau_dense.pt"
logreg = joblib.load(logreg_path)
dense = load_dense(dense_path)


def logreg_predict(x: torch.Tensor) -> np.ndarray:
    return logreg.predict(standardize(x, MEAN, STD).flatten(1).numpy())


def dense_predict(x: torch.Tensor) -> np.ndarray:
    return dense(standardize(x, MEAN, STD)).argmax(1).numpy()


def resnet_predict(x: torch.Tensor) -> np.ndarray:
    return reloaded(standardize(x, IMAGENET_MEAN, IMAGENET_STD)).argmax(1).numpy()


def latency_ms(predict_fn, x: torch.Tensor, n: int = 100) -> float:
    """Temps médian d'une prédiction pour une seule œuvre uint8 (C, H, W), en millisecondes."""
    predict_fn(x[None])  # premier appel hors mesure
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        predict_fn(x[None])
        times.append(time.perf_counter() - t0)
    return 1000 * float(np.median(times))


MODELS = {
    "régression logistique": (logreg_predict, logreg.coef_.size + logreg.intercept_.size, logreg_path),
    "réseau dense": (dense_predict, sum(p.numel() for p in dense.parameters()), dense_path),
    "ResNet-18 fine-tuné": (resnet_predict, sum(p.numel() for p in reloaded.parameters()), resnet_path),
}
X_val, y_val = X_all[splits["val"]], y_all[splits["val"]].numpy()
n_threads = torch.get_num_threads()
rows = []
with torch.no_grad():
    for name, (predict_fn, n_params, path) in MODELS.items():
        acc = (np.concatenate([predict_fn(X_val[i:i + 100]) for i in range(0, len(X_val), 100)]) == y_val).mean()
        ms_all = latency_ms(predict_fn, X_val[0])
        torch.set_num_threads(1)
        ms_one = latency_ms(predict_fn, X_val[0])
        torch.set_num_threads(n_threads)
        rows.append({"modele": name, "acc_val": acc, "ic95": 1.96 * math.sqrt(acc * (1 - acc) / len(y_val)),
                     f"ms_{n_threads}_coeurs": ms_all, "ms_1_coeur": ms_one,
                     "parametres": n_params, "taille_mo": path.stat().st_size / 1e6})

comparison = pd.DataFrame(rows).set_index("modele")
print(f"Mesuré sur CPU, {n_threads} cœurs puis 1 : {platform.platform()}")
comparison.round({"acc_val": 3, "ic95": 3, f"ms_{n_threads}_coeurs": 2, "ms_1_coeur": 2, "taille_mo": 1})

- **Le ResNet-18 fine-tuné écrase les deux modèles sur pixels** : 0,837 en validation, contre 0,427 pour la régression logistique et 0,410 pour le réseau dense. Les intervalles de confiance à 95 % (± 0,042 et ± 0,056) sont loin de se chevaucher.
- **Il est beaucoup plus lent sur CPU** : quelques dizaines de millisecondes par œuvre, contre une milliseconde environ ou moins pour les deux autres modèles. Cela reste rapide pour un visiteur, qui attend sa réponse moins d'un dixième de seconde, même sur un seul cœur. Ces temps varient d'une exécution à l'autre et sont mesurés sur le processeur d'un Mac M1 : une borne modeste sera plus lente, d'où la mesure sur un seul cœur.
- **Et plus lourd** : 11 179 077 paramètres et 44,8 Mo, contre 6 292 229 paramètres et 25,2 Mo pour le réseau dense, 245 765 paramètres et 1,0 Mo pour la régression logistique.
- **Le réseau dense n'a aucun avantage** : moins précis que la régression logistique, plus lent et plus lourd qu'elle.

## 5. Bilan

- **Le ResNet-18 pré-entraîné l'emporte nettement** : 0,780 en validation avec la tête seule, 0,838 ± 0,015 après le fine-tuning de `layer4`, contre 0,427 pour la régression logistique, le meilleur des modèles sur pixels. L'hypothèse de `03_diagnostic` se vérifie : des motifs locaux appris sur ImageNet séparent les styles bien mieux que la luminosité et les couleurs d'ensemble.
- **Le fine-tuning apporte un gain réel mais demande un arrêt précoce** : le témoin, la tête seule entraînée plus longtemps, reste à 0,799 ± 0,008, et le train approche 1 dès les premières époques de fine-tuning.
- **Coût pour les bornes** : quelques dizaines de millisecondes par œuvre sur un seul cœur et 44,8 Mo, contre moins d'une milliseconde et 1,0 Mo pour la régression logistique. `06_recommandation` pèsera ce coût contre le gain de précision, sur le test.
- **Le modèle fine-tuné est sauvegardé** (44,8 Mo) pour l'évaluation finale.